# Answer / Critique / Refine Workflow — Deployed to Agent Platform

Builds the same search -> critique -> refine ADK pipeline as before, then
deploys it to **Vertex AI Agent Engine** so it runs as a standalone remote
service instead of only inside this notebook:

1. Sections 1-8 (unchanged) build `greeter`, the root agent of the
   search/critique/refine pipeline, and test it locally.
2. Section 9 initializes the `vertexai` SDK with a staging bucket, then
   wraps `answer_team` (the `SequentialAgent` itself, rather than
   `greeter`) in an `AdkApp` and confirms it still runs correctly through
   that wrapper before deploying anything. `greeter`'s only job was to
   unconditionally forward every question to `answer_team`, so deploying
   `answer_team` directly is a harmless simplification either way.
3. Section 10 deploys that `AdkApp` to Agent Platform with
   `agent_engines.create()` — this provisions a real, billable remote
   resource and can take several minutes. It pins the deploy
   `requirements` to the exact `google-adk`/`google-cloud-aiplatform`
   versions installed in this notebook: without that, the deployed
   container resolves whatever versions pip picks at build time, which can
   silently differ from what's running locally (where everything works)
   and has been observed to cause a `TypeError: 'NoneType' object is not
   subscriptable` once deployed and queried remotely — even for agent
   trees that run fine locally through both `Runner` and `AdkApp`.
4. Section 11 sends a test query to the **deployed** agent (not the local
   one) to confirm the remote deployment actually works.
5. Section 12 is an optional cleanup cell that deletes the remote
   deployment when you're done with it.


## 1. Install dependencies

In [13]:
%pip install --quiet google-adk "google-cloud-aiplatform[agent_engines,adk]"


## 2. Configuration

This notebook only calls Gemini (no weather tools, no Maps key, no
third-party model), so the only setup needed is Vertex AI auth using
this notebook's existing GCP credentials — no personal key required.

In [14]:
import os

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ.setdefault("GOOGLE_CLOUD_LOCATION", "us-central1")

if "GOOGLE_CLOUD_PROJECT" not in os.environ:
    import subprocess

    try:
        _detected_project = subprocess.check_output(
            ["gcloud", "config", "get-value", "project"],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
        if _detected_project and _detected_project != "(unset)":
            os.environ["GOOGLE_CLOUD_PROJECT"] = _detected_project
    except (subprocess.CalledProcessError, FileNotFoundError):
        pass

print("Vertex project:", os.environ.get("GOOGLE_CLOUD_PROJECT"))
print("Vertex location:", os.environ.get("GOOGLE_CLOUD_LOCATION"))


Vertex project: qwiklabs-gcp-03-18ae669cea60
Vertex location: us-central1


## 3. Resolve an available Gemini model

Tries a shortlist of Gemini model/region combinations on Vertex AI and uses
the first one this project actually has access to.


In [15]:
from google import genai
from google.genai.errors import ClientError

_GEMINI_MODEL_CANDIDATES = [
    "gemini-2.5-flash-lite",  # confirmed available in Model Garden
    "gemini-2.5-flash",
    "gemini-2.5-pro",
    "gemini-3.5-flash-lite",
    "gemini-2.0-flash-001",
    "gemini-2.0-flash",
    "gemini-1.5-flash-002",
]
_GEMINI_REGION_CANDIDATES = ["us-central1", "us-east4", "us-east5", "us-west1", "europe-west4"]

_project = os.environ["GOOGLE_CLOUD_PROJECT"]
MODEL_ID = None
GEMINI_LOCATION = None

for _region in _GEMINI_REGION_CANDIDATES:
    _candidate_client = genai.Client(vertexai=True, project=_project, location=_region)
    for _model in _GEMINI_MODEL_CANDIDATES:
        try:
            _candidate_client.models.generate_content(model=_model, contents="ping")
        except ClientError as exc:
            if exc.code == 404:
                print(f"unavailable: {_model} in {_region} (404)")
                continue
            raise
        MODEL_ID = _model
        GEMINI_LOCATION = _region
        break
    if MODEL_ID:
        break

if MODEL_ID is None:
    raise RuntimeError(
        "No Gemini model/region combination on Vertex AI worked for "
        f"project {_project}. Check Vertex AI > Model Garden in the "
        "console for what's actually enabled and add it to "
        "_GEMINI_MODEL_CANDIDATES/_GEMINI_REGION_CANDIDATES above."
    )

os.environ["GOOGLE_CLOUD_LOCATION"] = GEMINI_LOCATION
print(f"Using Gemini model: {MODEL_ID} in {GEMINI_LOCATION}")


Using Gemini model: gemini-2.5-flash-lite in us-central1


## 4. Search agent

Drafts an initial answer using ADK's built-in Google Search tool.
`output_key="initial_answer"` tells ADK to save this agent's final
response text into `session.state["initial_answer"]` automatically, so the
next agent in the pipeline can reference it.


In [16]:
from google.adk.agents import Agent
from google.adk.tools import google_search

search_agent = Agent(
    name="search_agent",
    model=MODEL_ID,
    description="Finds up-to-date information via Google Search to draft an initial answer.",
    instruction="""Use the google_search tool to find current, accurate
information that answers the user's question. Write a clear, well
organized initial answer based on what you find. This is a first draft —
it will be reviewed and improved by other agents next, so focus on
getting the facts right rather than polishing the wording.""",
    tools=[google_search],
    output_key="initial_answer",
)


## 5. Critique agent

Reviews the initial answer and lists concrete improvements — it does not
rewrite anything itself. Its instruction references `{initial_answer}`,
which ADK substitutes from session state at runtime. Its own output is
saved to `session.state["critique"]` via `output_key`.


In [17]:
critique_agent = Agent(
    name="critique_agent",
    model=MODEL_ID,
    description="Reviews the initial answer and suggests concrete improvements.",
    instruction="""You are a critical reviewer. Read the initial answer
below and identify concrete ways it could be improved: missing
information, unclear wording, unsupported claims, or organization
problems. If it's already solid, say so briefly. Do not rewrite the
answer yourself — only list specific, actionable suggestions.

Initial answer:
\"\"\"
{initial_answer}
\"\"\"
""",
    output_key="critique",
)


## 6. Refine agent

Rewrites the initial answer, applying the critique's suggestions, and
produces the final answer the user actually sees. Its own output is saved
to `session.state["final_answer"]`.


In [18]:
refine_agent = Agent(
    name="refine_agent",
    model=MODEL_ID,
    description="Rewrites the initial answer using the critique's suggestions.",
    instruction="""Rewrite the answer below, applying every applicable
suggestion from the critique. Produce a single polished, final answer for
the user — do not mention the review process, the critique, or any other
agent in your output; just answer the original question well.

Initial answer:
\"\"\"
{initial_answer}
\"\"\"

Critique / suggested improvements:
\"\"\"
{critique}
\"\"\"
""",
    output_key="final_answer",
)


## 7. Answer team (SequentialAgent) and greeter (root agent)

`SequentialAgent` runs `search_agent`, `critique_agent`, then
`refine_agent` in that fixed order — no LLM decides the order, so there's
no risk of the transfer/tool conflict a dynamically-routing root agent
would have with `google_search` (mixing a built-in search tool with ADK's automatic
`transfer_to_agent` injection trips a Vertex API restriction: here,
`search_agent`'s direct parent is a workflow agent, not an `LlmAgent` that
needs its own transfer tool). `greeter` is the entry point a user
actually talks to; it has no tools of its own and simply hands every
question to the answer team as a sub agent.


In [19]:
from google.adk.agents import SequentialAgent

answer_team = SequentialAgent(
    name="answer_team",
    description="Answers a question, critiques the initial answer, then refines it.",
    sub_agents=[search_agent, critique_agent, refine_agent],
)

greeter = Agent(
    name="greeter",
    model=MODEL_ID,
    description="Entry point that forwards user questions to the answer team.",
    instruction="""You are the entry point for a question-answering
assistant. For every question the user asks, delegate immediately to the
answer team rather than answering yourself.""",
    sub_agents=[answer_team],
)


/tmp/ipykernel_55432/1604226314.py:3: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  answer_team = SequentialAgent(


## 8. Test harness

Runs a question through `greeter` and prints the raw event stream — each
event's `author` shows which agent (`greeter`, `search_agent`,
`critique_agent`, or `refine_agent`) produced it, so you can see the whole
answer -> critique -> refine pipeline execute step by step, not just the
final text. It then also prints the session's final state, showing all
three stored outputs side by side.


In [20]:
import asyncio

from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types as genai_types

APP_NAME = "answer_workflow_app"
USER_ID = "test_user"

session_service = InMemorySessionService()


async def run_and_print_events(agent: Agent, query: str, session_id: str) -> None:
    """Run one query through an ADK agent and print every event it emits.

    Args:
        agent: The (root) agent to run the query through.
        query: The user message to send.
        session_id: A unique session id for this run.
    """
    await session_service.create_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=session_id
    )
    runner = Runner(agent=agent, app_name=APP_NAME, session_service=session_service)
    content = genai_types.Content(role="user", parts=[genai_types.Part(text=query)])

    print(f"\n--- Query: {query} ---")
    async for event in runner.run_async(
        user_id=USER_ID, session_id=session_id, new_message=content
    ):
        author = getattr(event, "author", "?")
        parts = event.content.parts if event.content else []
        for part in parts:
            if getattr(part, "function_call", None):
                fc = part.function_call
                print(f"[{author}] FUNCTION_CALL: {fc.name}({dict(fc.args or {})})")
            if getattr(part, "function_response", None):
                fr = part.function_response
                print(f"[{author}] FUNCTION_RESPONSE: {fr.name} -> (truncated)")
            if getattr(part, "text", None):
                tag = "FINAL" if event.is_final_response() else "TEXT"
                print(f"[{author}] {tag}: {part.text.strip()[:400]}")

    session = await session_service.get_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=session_id
    )
    print("\n--- Session state after the run ---")
    for key in ("initial_answer", "critique", "final_answer"):
        print(f"\n[{key}]\n{session.state.get(key)}")


TEST_QUERY = "What are the main new capabilities in the latest Gemini model family?"


async def run_tests() -> None:
    await run_and_print_events(greeter, TEST_QUERY, session_id="greeter-session-0")


await run_tests()



--- Query: What are the main new capabilities in the latest Gemini model family? ---
[greeter] FUNCTION_CALL: transfer_to_agent({'agent_name': 'answer_team'})
[greeter] FUNCTION_RESPONSE: transfer_to_agent -> (truncated)
[search_agent] FINAL: The latest Gemini model family boasts several significant new capabilities, enhancing its ability to process and understand information across multiple modalities, extending its context window, and improving its reasoning and agentic functionalities.

Key advancements include:

*   **Expanded Multimodal Capabilities:** Gemini models can now process and understand text, images, audio, and video. Ge
[critique_agent] FINAL: The answer provides a good overview of the Gemini model family's new capabilities. However, it could be improved in the following ways:

*   **Specificity for Gemini 2.0:** While Gemini 1.5 Pro is mentioned for context window and multimodal capabilities, some points, like "Enhanced Reasoning and Agentic AI" and "Native Tool Use a

## 9. Initialize Vertex AI and test locally via AdkApp

Deploying to Agent Platform uses a different SDK entry point than the
direct Gemini calls above: the `vertexai` package's `agent_engines` module,
which needs its own `vertexai.init()` call with a **Cloud Storage staging
bucket** (used to package up the agent code for deployment). The cell below
reuses the project/location already resolved in Section 3, creates a
staging bucket if one doesn't already exist, then wraps `answer_team` in
an `AdkApp` and runs the same kind of local test as Section 8 through
that wrapper, confirming the exact object that's about to be deployed
still behaves correctly before spending the time deploying it.


In [21]:
import subprocess

import vertexai
from vertexai.preview import reasoning_engines

STAGING_BUCKET_NAME = f"{_project}-agent-engine-staging"
STAGING_BUCKET = f"gs://{STAGING_BUCKET_NAME}"

# Create the staging bucket if it doesn't already exist.
_bucket_check = subprocess.run(
    ["gsutil", "ls", "-b", STAGING_BUCKET], capture_output=True, text=True
)
if _bucket_check.returncode != 0:
    subprocess.run(
        ["gsutil", "mb", "-l", GEMINI_LOCATION, STAGING_BUCKET], check=True
    )
    print(f"Created staging bucket: {STAGING_BUCKET}")
else:
    print(f"Using existing staging bucket: {STAGING_BUCKET}")

vertexai.init(
    project=_project,
    location=GEMINI_LOCATION,
    staging_bucket=STAGING_BUCKET,
)

# Deploy answer_team directly rather than greeter: greeter always
# forwarded to answer_team unconditionally, so this is a harmless
# simplification (not a bugfix — see Section 10 for the actual fix).
app = reasoning_engines.AdkApp(agent=answer_team)

# AdkApp needs an explicit session before stream_query — without one it
# has no session to attach state/events to and raises a TypeError
# ("'NoneType' object is not subscriptable") on the very first turn.
# create_session() returns a plain dict here, not an object, so index it
# with ["id"] rather than .id.
session = app.create_session(user_id="local-test-user")

print("\n--- Local test via AdkApp ---")
for event in app.stream_query(
    user_id="local-test-user",
    session_id=session["id"],
    message="What are the main new capabilities in the latest Gemini model family?",
):
    print(event)


Using existing staging bucket: gs://qwiklabs-gcp-03-18ae669cea60-agent-engine-staging

--- Local test via AdkApp ---


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:966: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


{'model_version': 'gemini-2.5-flash-lite', 'content': {'parts': [{'text': "The latest Gemini model family introduces several new capabilities, focusing on enhanced multimodal understanding, longer context windows, improved reasoning, and more agentic features.\n\nKey advancements include:\n\n*   **Expanded Multimodal Capabilities:** Gemini models can now process and generate content across various modalities, including text, images, audio, and video. This allows for more integrated and dynamic interactions, enabling applications to understand and produce diverse forms of information. Gemini 2.0, for instance, supports multimodal output, generating content that mixes images, text, and controllable text-to-speech audio.\n*   **Significantly Increased Context Windows:** Gemini 1.5 Pro boasts a context window of up to 1 million tokens, with potential scalability up to 2 million tokens. This extensive context window allows Gemini to analyze and understand vast amounts of data, such as large

## 10. Deploy to Agent Platform

`agent_engines.create()` packages the `AdkApp` (and its dependencies) and
deploys it as a managed remote agent. This actually provisions cloud
infrastructure — expect this cell to take several minutes, and note that
the deployed agent is a billable resource until it's deleted (Section 12).

The `requirements` list is pinned to the exact `google-adk` and
`google-cloud-aiplatform` versions installed in this notebook (Section 1),
rather than left unpinned. An unpinned `google-cloud-aiplatform[adk]`
resolves whatever version pip picks at build time in the deployed
container, which can differ from what's actually running in this kernel
— and that mismatch has been observed to produce a
`TypeError: 'NoneType' object is not subscriptable` once the agent is
deployed and queried remotely, even though the identical agent runs fine
locally through both `Runner` and `AdkApp`.


In [22]:
import subprocess

from vertexai import agent_engines


def _installed_version(package: str) -> str:
    """Return the installed version of a pip package in this kernel.

    Args:
        package: The pip package name to look up.

    Returns:
        The installed version string.

    Raises:
        RuntimeError: If the package isn't installed or has no
            discoverable version.
    """
    output = subprocess.run(
        ["pip", "show", package], capture_output=True, text=True
    ).stdout
    for line in output.splitlines():
        if line.startswith("Version:"):
            return line.split(":", 1)[1].strip()
    raise RuntimeError(f"Could not determine installed version of {package}")


_adk_version = _installed_version("google-adk")
_aiplatform_version = _installed_version("google-cloud-aiplatform")
print(f"Pinning deploy requirements to: google-adk=={_adk_version}, "
      f"google-cloud-aiplatform=={_aiplatform_version}")

remote_agent = agent_engines.create(
    app,
    requirements=[
        f"google-adk=={_adk_version}",
        f"google-cloud-aiplatform[agent_engines,adk]=={_aiplatform_version}",
    ],
)

print("Deployed agent resource name:", remote_agent.resource_name)


INFO:vertexai.agent_engines:Identified the following requirements: {'pydantic': '2.13.4', 'google-cloud-aiplatform': '1.163.0', 'cloudpickle': '3.1.2'}
INFO:vertexai.agent_engines:The following requirements are appended: {'pydantic==2.13.4', 'cloudpickle==3.1.2'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-adk==2.4.0', 'google-cloud-aiplatform[agent_engines,adk]==1.163.0', 'pydantic==2.13.4', 'cloudpickle==3.1.2']
INFO:vertexai.agent_engines:Using bucket qwiklabs-gcp-03-18ae669cea60-agent-engine-staging


Pinning deploy requirements to: google-adk==2.4.0, google-cloud-aiplatform==1.163.0


INFO:vertexai.agent_engines:Wrote to gs://qwiklabs-gcp-03-18ae669cea60-agent-engine-staging/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-03-18ae669cea60-agent-engine-staging/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-03-18ae669cea60-agent-engine-staging/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing LRO: projects/636697947440/locations/us-central1/reasoningEngines/4045716256320389120/operations/5033714061892648960
INFO:vertexai.agent_engines:View progress and logs at https://console.cloud.google.com/logs/query?project=qwiklabs-gcp-03-18ae669cea60
INFO:vertexai.agent_engines:AgentEngine created. Resource name: projects/636697947440/locations/us-central1/reasoningEngines/4045716256320389120
INFO:vertexai.agent_engines:To use this AgentEngine i

Deployed agent resource name: projects/636697947440/locations/us-central1/reasoningEngines/4045716256320389120


## 11. Test the deployed agent

Sends a query to the **remote** deployed agent (not the local
`answer_team` object) to confirm the actual deployment works, not just
the code that produced it. Uses `async_stream_query` rather than
`stream_query`: the synchronous `stream_query` has been observed to hang
or silently return zero events on a freshly deployed Agent Engine
resource even when the agent runs correctly, while `async_stream_query`
reliably returns the actual events.


In [ ]:
def _print_remote_event(event: dict) -> None:
    """Print one raw event dict from a deployed AdkApp in a readable form.

    Deployed/remote queries return plain dicts rather than the event
    objects `Runner.run_async` yields locally, so this mirrors the
    formatting used in Section 8's local test harness but reads fields
    with `dict.get` instead of attribute access. Falls back to printing
    the raw dict for any event that doesn't match the shapes below
    (e.g. an error event), so nothing is ever silently dropped.

    Args:
        event: One event dict yielded by the remote query.
    """
    parts = (event.get("content") or {}).get("parts") or []
    author = event.get("author", "?")
    printed_something = False
    for part in parts:
        if "function_call" in part:
            fc = part["function_call"]
            print(f"[{author}] FUNCTION_CALL: {fc['name']}({fc.get('args', {})})")
            printed_something = True
        elif "function_response" in part:
            fr = part["function_response"]
            print(f"[{author}] FUNCTION_RESPONSE: {fr['name']} -> {fr.get('response')}")
            printed_something = True
        elif part.get("text"):
            print(f"[{author}] TEXT: {part['text'].strip()}")
            printed_something = True

    if not printed_something:
        # Unrecognized shape (e.g. an error event with no "content"/"parts") —
        # print the raw dict rather than silently dropping it.
        print(f"[RAW EVENT] {event}")


print("--- Remote test via deployed agent ---")
print("Deployed resource:", remote_agent.resource_name)

remote_session = remote_agent.create_session(user_id="agent-engine-test-user")


async def _run_remote_test() -> None:
    events = []
    async for event in remote_agent.async_stream_query(
        user_id="agent-engine-test-user",
        session_id=remote_session["id"],
        message="What is the National Weather Service and what does it cover?",
    ):
        events.append(event)
        _print_remote_event(event)
    print(f"\nGot {len(events)} events")


await _run_remote_test()


--- Remote test via deployed agent ---
Deployed resource: projects/636697947440/locations/us-central1/reasoningEngines/4045716256320389120
[search_agent] TEXT: The National Weather Service (NWS) is a United States federal government agency responsible for providing weather forecasts, issuing warnings for hazardous weather, and delivering other weather-related products. It is a part of the National Oceanic and Atmospheric Administration (NOAA), which itself is a branch of the Department of Commerce. The NWS headquarters are located in Silver Spring, Maryland.

The primary mission of the NWS is to protect lives and property and to enhance the national economy by providing crucial weather, water, and climate information. This includes issuing official warnings for life-threatening weather situations, making it the sole voice of the U.S. government in such instances.

The NWS operates through a network of national and regional centers, along with 122 local Weather Forecast Offices (WFOs) s

## 12. Clean up (optional)

Deployed Agent Platform resources keep running (and billing) until deleted.
Uncomment and run this cell once you're done testing the deployment.


In [ ]:
# remote_agent.delete()
# print("Deleted:", remote_agent.resource_name)
